## Read embeddings

In [53]:
import pandas as pd
# MODELS PARAMS
total_splits = 1
MODELS_DESCRIPTIONS = {
           "class_I_simple_clustering": {
        "path": "experiment_results/paper2_core_clustering/reconstructed_models/",
        "states": 15,
        "params_string": "60-min_iters-4-aaas",
        "combination_size": 2,
        "alleles": [
                # 'HLA-DRB1*01:01'
                    'HLA-A*01:01',
                    'HLA-A*03:01'
                    ],
        "target_lengths": [8, 9,10, 11, 12,13, 14],
        "core_start_state": [15,16],
        "core_end_state": [17,18],
        "need_to_delete": [],
        "skip_states": [],

    },
                   "class_I_simple_clustering_simulated": {
        "path": "experiment_results/paper2_core_clustering/reconstructed_models/",
        "states": 15,
        "params_string": "60-min_iters-4-aaas",
        "combination_size":1,
        "alleles": [
             "dummy_allele"
                # 'HLA-DRB1*01:01'

                    ],
        "target_lengths": [ 9,10, 11],
        "core_start_state": [15,16],
        "core_end_state": [17,18],
        "need_to_delete": [],
        "skip_states": [],

    },
}

# selected_model = MODELS_DESCRIPTIONS["multiple_allele_imbalanced"]
# selected_model = MODELS_DESCRIPTIONS["multiple_allele_balanced"]
# selected_model = MODELS_DESCRIPTIONS["single_allele_imbalanced"]
# selected_model = MODELS_DESCRIPTIONS["single_allele_balanced"]
selected_model = MODELS_DESCRIPTIONS["class_I_simple_clustering_simulated"]



In [54]:
#Selected combination
# combination = ('HLA-A*01:01', 'HLA-A*02:01')
#combination = ('HLA-A*01:01', 'HLA-A*03:01')
combination = ('dummy_allele',)
# combination = ('HLA-A*02:01', 'HLA-A*03:01')
# combination = ('HLA-A*01:01','HLA-A*02:01', 'HLA-A*03:01')

# combination = ('HLA-A*01:01',)
# combination = ('HLA-A*03:01',)
# combination = ('HLA-A*02:01',)
# combination = ('HLA-DRB1*01:01', 'HLA-DPA1*01:03/DPB1*04:01')
# combination = ('HLA-DRB1*01:01',)
#combination = ('HLA-DQA1*05:01/DQB1*02:01',)

In [55]:
combination

('dummy_allele',)

In [56]:
selected_model['alleles']

['dummy_allele']

In [57]:
import numpy as np
for allele_name in combination:
    assert allele_name in selected_model['alleles'], f"{allele_name} not in "
selected_alleles = list(combination)


states = selected_model['states']
TARGET_LENGTHS = selected_model['target_lengths']
selected_combination = "+".join(sorted(selected_alleles)).replace('-', '_').replace('*', '_').replace(':', '_').replace("/", "_")
selected_model_split = 0 # should match to data split (if original train/test used) in order to have true test case
# selected_model_run = 0
target_length_string = "-".join([str(l) for l in TARGET_LENGTHS])

RUNS = 20

SELECTED_RUNS = set(np.arange(0, RUNS))


models_to_read = {'v'}

# Join runs into single file

In [58]:
from collections import defaultdict
per_dataset_per_model_type_embeddings = defaultdict(dict)

name_to_model_root = dict()

def load_embeddings(model, dataset_type="train"):
    print(f"model {model}")
    TARGET_PATH_TO_FILES = name_to_model_root[model] + model
    # Train 
    train_df =  pd.read_csv("{}/{}_embedding.csv".format(TARGET_PATH_TO_FILES, dataset_type),sep=";",index_col=0)
    # paths_df['path'] = paths_df['path'].apply(pd.eval)
    # paths_df['path'] = paths_df['path'].astype(list)
    return  train_df

def read_multiple_runs_to_single_df(model_type, dataset_type, index_columns=['allele', 'peptide']):
    for run_num in SELECTED_RUNS:
        print("Run", run_num, ":", end="")
        print(f"\t{model_type}",)
        params_string = selected_model['params_string']
        if model_type == 'bw':

            mixed_model_name =f'{run_num}_run-{target_length_string}-{states}-st-baum-welch-alg-{params_string}_model-{selected_combination}-{selected_model_split}'
        elif model_type == 'v':
            mixed_model_name = f'{run_num}_run-{target_length_string}-{states}-st-viterbi-alg-{params_string}_model-{selected_combination}-{selected_model_split}'
        elif model_type == 'bwv':
            mixed_model_name = f'{run_num}_run-{target_length_string}-{states}-st-baum-welch-alg-{params_string}_model-{selected_combination}-{selected_model_split}_plus_viterbi'
        else:
            raise Exception("unknown model type", model_type)
        
        name_to_model_root[mixed_model_name] = selected_model['path']
        train_df = load_embeddings(mixed_model_name, dataset_type)
        train_df.reset_index(inplace=True, drop=True)

        print("Initial shape train:", train_df.shape)
        
        if run_num == 0:
            # Train
            target_train_df = train_df
            target_train_df.sort_values(by=index_columns)
            saved_index_columns = target_train_df[index_columns].copy()
            target_train_df= target_train_df.drop(columns = index_columns)
        else:
            #Train
            target_train_df = per_dataset_per_model_type_embeddings[dataset_type][model_type]
            train_df.sort_values(by=index_columns)
            train_df = train_df.drop(columns = index_columns)
            target_train_df = pd.concat([target_train_df,train_df], axis=1)
            target_train_df.columns = [i for i in range (len(target_train_df.columns))]
        print("Shape of merged df:", target_train_df.shape)
        
        if run_num == max(SELECTED_RUNS):
            target_train_df[index_columns] = saved_index_columns

        per_dataset_per_model_type_embeddings[dataset_type][model_type] = target_train_df
    

    
    

In [59]:
if "bw" in models_to_read:
    read_multiple_runs_to_single_df('bw', 'train')
    # read_multiple_runs_to_single_df('bw', 'test')
if "v" in models_to_read:
    read_multiple_runs_to_single_df('v', 'train')
    # read_multiple_runs_to_single_df('v', 'test')
if "bwv" in models_to_read:
    read_multiple_runs_to_single_df('bwv', 'train')
    # read_multiple_runs_to_single_df('bwv', 'test')

    

Run 0 :	v
model 0_run-9-10-11-15-st-viterbi-alg-60-min_iters-4-aaas_model-dummy_allele-0
Initial shape train: (11999, 502)
Shape of merged df: (11999, 500)
Run 1 :	v
model 1_run-9-10-11-15-st-viterbi-alg-60-min_iters-4-aaas_model-dummy_allele-0
Initial shape train: (11999, 502)
Shape of merged df: (11999, 1000)
Run 2 :	v
model 2_run-9-10-11-15-st-viterbi-alg-60-min_iters-4-aaas_model-dummy_allele-0
Initial shape train: (11999, 502)
Shape of merged df: (11999, 1500)
Run 3 :	v
model 3_run-9-10-11-15-st-viterbi-alg-60-min_iters-4-aaas_model-dummy_allele-0
Initial shape train: (11999, 502)
Shape of merged df: (11999, 2000)
Run 4 :	v
model 4_run-9-10-11-15-st-viterbi-alg-60-min_iters-4-aaas_model-dummy_allele-0
Initial shape train: (11999, 502)
Shape of merged df: (11999, 2500)
Run 5 :	v
model 5_run-9-10-11-15-st-viterbi-alg-60-min_iters-4-aaas_model-dummy_allele-0
Initial shape train: (11999, 502)
Shape of merged df: (11999, 3000)
Run 6 :	v
model 6_run-9-10-11-15-st-viterbi-alg-60-min_iter

# Save joined embeddings to separate folder

In [61]:
import os
def save_embeddings(model_type, dataset_type):
    if model_type == 'bw':
        mixed_model_name =f'combined_run-{target_length_string}-{states}-st-baum-welch-alg-5-min_iters_model-{selected_combination}-{selected_model_split}'
    elif model_type == 'v':
        mixed_model_name = f'combined_run-{target_length_string}-{states}-st-viterbi-alg-5-min_iters_model-{selected_combination}-{selected_model_split}'
    elif model_type == 'bwv':
        mixed_model_name = f'combined_run-{target_length_string}-{states}-st-baum-welch-alg-5-min_iters_model-{selected_combination}-{selected_model_split}_plus_viterbi'
    else:
        raise Exception("unknown model type", model_type)
    
    model_root = f"{selected_model['path']}{mixed_model_name}"
    if not os.path.exists(model_root):
        os.makedirs(model_root)
    embedding = per_dataset_per_model_type_embeddings[dataset_type][model_type]
    embedding.to_csv("{}/{}_embedding.csv".format(model_root, dataset_type),sep=";", index=True)
        
    

In [62]:
print(1)
if "bw" in models_to_read:
    save_embeddings('bw', 'train')
    # save_embeddings('bw', 'test')
print(2)

if "v" in models_to_read:
    save_embeddings('v', 'train')
    # save_embeddings('v', 'test')
print(3)
if "bvw" in models_to_read:
    save_embeddings('bwv', 'train')
    # save_embeddings('bwv', 'test')

1
2
3


In [63]:
per_dataset_per_model_type_embeddings['train']['v']

,0,1,2,3,4,5,6,7,8,9,...,9992,9993,9994,9995,9996,9997,9998,9999,allele,peptide
0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Dummy_allele_4,FDGIKYEERL
1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Dummy_allele_4,FWWLKQYRRL
2,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Dummy_allele_4,VSHLMDETYL
3,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Dummy_allele_4,FWNPKMEMTV
4,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Dummy_allele_5,VMHVSKTWFQ
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
11994,NaN,-35.036187,-44.101359,-30.628897,-31.167951,-34.994206,-38.705498,-42.726981,-38.775937,-33.976056,...,NaN,NaN,-32.669488,NaN,NaN,NaN,NaN,-31.36065,Dummy_allele_8,FRQTAHNPL
11995,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,-43.106045,NaN,NaN,NaN,Dummy_allele_2,FARSLHTEFCN
11996,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Dummy_allele_5,VFHKTFGRLN
11997,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Dummy_allele_5,VTKRDKGSEQ
